# 10 · Query Expansion

Generate synonym/paraphrase queries to raise recall (then fuse or rerank).

**Analogy handbook:** [expansion](../retriever-analogy-handbook.html#expansion)  
**Prerequisite:** run `00_basics_concepts.ipynb` once (or the setup cells below) so the Chroma index exists.

### Learning loop
1. Skim the analogy for this technique  
2. Run setup (reuse index if possible)  
3. Run the practical cells  
4. Ask: *Did this fix the failure mode, or only reshuffle noise?*


## Shared setup

These cells install packages, load the Llama 2 PDF, build/load the Chroma index, and define helpers.

> Prefer `REBUILD_INDEX = False` after the first successful build so later method notebooks reuse the same store.


### Learning: !pip install langchain_community langchain_text_splitters langchain_op

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Installs required Python packages into the runtime.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
!pip install langchain_community langchain_text_splitters langchain_openai langchain_chroma pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 2.2 MB/s eta 0:00:00


### Learning: IMPORTS

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `IMPORTS` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
print("All imports and setup starting...")

# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import getpass
import os
import shutil

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_classic.chains.hyde.base import (
    HypotheticalDocumentEmbedder
)

All imports and setup starting...


/tmp/ipykernel_520/2286258816.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Learning: from google.colab import userdata

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `from google.colab import userdata` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

### Learning: OPENAI API KEY

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `OPENAI API KEY` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 2. OPENAI API KEY
# ============================================================

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured successfully.")

OpenAI API key configured successfully.


### Learning: DATA DIRECTORY

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `DATA DIRECTORY` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 3. DATA DIRECTORY
# ============================================================

DATA_DIR = Path(
    r"/content/"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

### Learning: FIND PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `FIND PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 4. FIND PDF
# ============================================================

if preferred_pdf.exists():

    PDF_PATH = preferred_pdf

else:

    available_pdfs = sorted(
        DATA_DIR.glob("*.pdf")
    )

    if len(available_pdfs) == 1:

        PDF_PATH = available_pdfs[0]

    elif len(available_pdfs) == 0:

        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )

    else:

        raise RuntimeError(
            "Multiple PDF files were found. "
            "Please set PDF_PATH manually.\n"
            + "\n".join(
                str(path)
                for path in available_pdfs
            )
        )


print("PDF found:")
print(PDF_PATH)

PDF found:
/content/llama2-research-paper.pdf


### Learning: LOAD PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `LOAD PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [15]:
# ============================================================
# 5. LOAD PDF
# ============================================================

loader = PyPDFLoader(
    str(PDF_PATH)
)

pages = loader.load()

print(
    f"\nTotal PDF pages loaded: {len(pages)}"
)


# ============================================================
# 6. INSPECT FIRST PAGE
# ============================================================

print("\nFirst-page metadata:")
print(
    pages[0].metadata
)

print("\nFirst 1,000 characters:")
print(
    pages[0].page_content[:1000]
)


Total PDF pages loaded: 77

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Art

### Learning: IDENTIFY PAPER SECTIONS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Defines helper logic for: IDENTIFY PAPER SECTIONS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. IDENTIFY PAPER SECTIONS
# ============================================================

def identify_section(
    paper_page: int
) -> str:

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"


# ============================================================
# 8. ADD METADATA
# ============================================================

for page_document in pages:

    page_index = int(
        page_document.metadata.get(
            "page",
            0
        )
    )

    paper_page = (
        page_index + 1
    )

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(
                paper_page
            ),
            "access_level": "public",
        }
    )


print("\nMetadata after enrichment:")

for page_document in pages[:5]:

    print(
        page_document.metadata
    )



Metadata after enrichment:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-37-08-Aug-2026-prompting\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',

### Learning: TEXT SPLITTING

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEXT SPLITTING` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. TEXT SPLITTING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(
    pages
)

print(
    f"\nTotal pages: {len(pages)}"
)

print(
    f"Total chunks: {len(chunks)}"
)


# ============================================================
# 10. ADD CHUNK IDs
# ============================================================

for chunk_number, chunk in enumerate(
    chunks
):

    paper_page = chunk.metadata.get(
        "paper_page",
        "unknown"
    )

    chunk.metadata[
        "chunk_id"
    ] = (
        f"llama2-page-"
        f"{paper_page}-"
        f"chunk-{chunk_number}"
    )


print("\nFirst chunk content:")

print(
    chunks[0].page_content[:1000]
)

print("\nFirst chunk metadata:")

print(
    chunks[0].metadata
)


Total pages: 77
Total chunks: 343

First chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Shar

### Learning: CREATE EMBEDDING MODEL

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE EMBEDDING MODEL` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 11. CREATE EMBEDDING MODEL
# ============================================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# ============================================================
# 12. TEST EMBEDDING MODEL
# ============================================================

test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(
    f"\nEmbedding dimensions: "
    f"{len(test_vector)}"
)

print(
    f"First 10 values: "
    f"{test_vector[:10]}"
)



Embedding dimensions: 1536
First 10 values: [0.0027942657470703125, -0.0521240234375, -0.021087646484375, -0.055419921875, -0.026397705078125, 0.028961181640625, -0.002071380615234375, 0.034759521484375, -0.0164794921875, -0.0245208740234375]


### Learning: CHROMA CONFIGURATION

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CHROMA CONFIGURATION` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 13. CHROMA CONFIGURATION
# ============================================================

PERSIST_DIRECTORY = (
    DATA_DIR
    / "chroma_llama2_retriever"
)

COLLECTION_NAME = (
    "llama2_retriever_demo"
)


### Learning: CREATE OR LOAD VECTOR STORE

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CREATE OR LOAD VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 14. CREATE OR LOAD VECTOR STORE
# ============================================================

# True  = rebuild complete vector DB
# False = reuse existing vector DB

REBUILD_INDEX = True

### Learning: VERIFY VECTOR STORE

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `VERIFY VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
if REBUILD_INDEX:

    print(
        "\nRebuilding vector store..."
    )

    if PERSIST_DIRECTORY.exists():

        shutil.rmtree(
            PERSIST_DIRECTORY,
            ignore_errors=True
        )

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(
            PERSIST_DIRECTORY
        ),
        collection_configuration={
            "hnsw": {
                "space": "cosine"
            }
        },
    )

    print(
        "New vector store created."
    )

else:

    if not PERSIST_DIRECTORY.exists():

        print(
            "\nExisting vector DB "
            "not found."
        )

        print(
            "Creating a new vector store..."
        )

        vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
            collection_configuration={
                "hnsw": {
                    "space": "cosine"
                }
            },
        )

        print(
            "New vector store created."
        )

    else:

        print(
            "\nLoading existing "
            "vector store..."
        )

        vector_store = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
        )

        print(
            "Existing vector store "
            "loaded."
        )


# ============================================================
# 15. VERIFY VECTOR STORE
# ============================================================

stored_count = (
    vector_store
    ._collection
    .count()
)

print(
    f"\nStored chunks: "
    f"{stored_count}"
)

print(
    f"Persisted at: "
    f"{PERSIST_DIRECTORY}"
)



Rebuilding vector store...
New vector store created.

Stored chunks: 343
Persisted at: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-37-08-Aug-2026-prompting\data\chroma_llama2_retriever


### Learning: Query Expansion

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Query expansion involves generating multiple alternative versions of a user's query to broaden the search and retrieve more comprehensive results. This increases the chances of matching relevant documents that might use different terminology.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
class ExpandedQueryOutput(BaseModel):
    queries: List[str] = Field(
        description=(
            "Four alternative search queries expressing "
            "the same information need using different wording."
        )
    )

### Learning: query_expansion_llm = llm.with_structured_output(ExpandedQueryOutput)

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** We configure the LLM to output a structured Pydantic object `ExpandedQueryOutput`, ensuring the output is a list of alternative queries.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
query_expansion_llm = llm.with_structured_output(ExpandedQueryOutput)

### Learning: query_expansion_prompt = ChatPromptTemplate.from_messages(

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** This `ChatPromptTemplate` instructs the LLM to generate four alternative search queries using synonyms, technical terms, and alternative wording, without answering the query itself.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
query_expansion_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Generate four alternative search queries for the user's query.

Use:
- synonyms,
- related technical terms,
- abbreviations where appropriate,
- alternative wording.

Do not answer the query.
Each query must preserve the original intent.
""",
        ),
        (
            "human",
            "Original query: {query}",
        ),
    ]
)

### Learning: query_expansion_chain = (query_expansion_prompt| query_expansion_llm)

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Create a LangChain chain for query expansion by combining the prompt and the structured output LLM.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
query_expansion_chain = (query_expansion_prompt| query_expansion_llm)

### Learning: original_query = (

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Define the original query for which we want to generate expanded versions.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
original_query = (
    "How was Llama 2-Chat improved using human feedback?"
)


### Learning: expanded_output = query_expansion_chain.invoke(

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Invoke the `query_expansion_chain` to get the `ExpandedQueryOutput` object containing the alternative queries.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
expanded_output = query_expansion_chain.invoke(
    {
        "query": original_query
    }
)

### Learning: expanded_output

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Display the raw output of the `expanded_output` object.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
expanded_output

ExpandedQueryOutput(queries=['How was Llama 2-Chat enhanced through human feedback?', 'In what ways did human feedback improve Llama 2-Chat?', 'What improvements were made to Llama 2-Chat using human-in-the-loop feedback?', 'How did human feedback contribute to the development of Llama 2-Chat?'])

### Learning: all_expanded_queries = [

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Combine the original query with all the generated expanded queries into a single list.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
all_expanded_queries = [
    original_query,
    *expanded_output.queries,
]


### Learning: all_expanded_queries

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Display the complete list of queries, including the original and its expanded versions.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
all_expanded_queries

['How was Llama 2-Chat improved using human feedback?',
 'How was Llama 2-Chat enhanced through human feedback?',
 'In what ways did human feedback improve Llama 2-Chat?',
 'What improvements were made to Llama 2-Chat using human-in-the-loop feedback?',
 'How did human feedback contribute to the development of Llama 2-Chat?']

### Learning: print("Generated search queries:\n")

**What you'll learn:** Generate synonym/paraphrase queries to raise recall.

**What this cell does:** Print the generated search queries in a readable format.

**Watch for:** More queries → more noise; plan to rerank or fuse.



In [ ]:
print("Generated search queries:\n")

for number, query in enumerate(
    all_expanded_queries,
    start=1,
):
    print(f"{number}. {query}")

Generated search queries:

1. How was Llama 2-Chat improved using human feedback?
2. How was Llama 2-Chat enhanced through human feedback?
3. In what ways did human feedback improve Llama 2-Chat?
4. What improvements were made to Llama 2-Chat using human-in-the-loop feedback?
5. How did human feedback contribute to the development of Llama 2-Chat?


### Learning: expanded_query_documents = []

**What you'll learn:** Pretty-print retrieved chunks for comparison.

**What this cell does:** We iterate through all expanded queries, perform a dense retrieval for each, combine the results, and then deduplicate them to get a comprehensive set of unique relevant documents.

**Watch for:** Always log page/section/chunk_id when debugging retrieval.



In [ ]:
expanded_query_documents = []

for query in all_expanded_queries:
    current_documents = dense_retriever.invoke(query)
    expanded_query_documents.extend(current_documents)

expanded_query_documents = deduplicate_documents(
    expanded_query_documents
)

display_documents(
    expanded_query_documents,
    title="Query Expansion: Combined Unique Documents",
    max_documents=10,
)


Query Expansion: Combined Unique Documents

RANK: 1
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-14
----------------------------------------------------------------------------------------------------
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy
Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward modeling datain
parallel with model enhancements is crucial to ensure the reward models remain within d

RANK: 2
Paper page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-10
---------------------------------------------------------------------------------------------------